## ENMO calculation 
L'ENMO (Euclidean Norm Minus One) est calculé à partir des trois axes d'accélération (x, y et z).

La norme euclidienne permet d'obtenir la magnitude du vecteur d'accélération. On soustrait ensuite 1 g afin de retirer la contribution de la gravité.

ENMO = √(x² + y² + z²) - 1


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df= pd.read_csv("0_z.csv", comment="#")
df.head()

,t,x,y,z
0,0.00,-0.0938,-0.0156,0.9531
1,0.02,-0.0938,-0.0156,0.9531
2,0.04,-0.0938,-0.0156,0.9531
3,0.06,-0.0938,-0.0156,0.9531
4,0.08,-0.0938,-0.0156,0.9531


In [10]:
dt = df["t"].iloc[1]- df["t"].iloc[0]
fs = 1/dt

On a crée une fonction nous permettant de calculer l'ENMO :
celle-ci prend en entrée les valeurs des axes x,y,z et renvoie en sortie l'ENMO qui a été calculé. 

In [5]:
def calculate_enmo(x, y, z):
    return np.sqrt(x**2 + y**2 + z**2) - 1

Ici on décide de stocker les résultats des calculs précédents  dans une nouvelle colonne appelée "enmo" dans notre DataFrame df.

In [6]:
df["enmo"] = calculate_enmo(df["x"], df["y"], df["z"])

In [ ]:
df[["t", "enmo"]].head()

,t,enmo
0,0.00,-0.042168
1,0.02,-0.042168
2,0.04,-0.042168
3,0.06,-0.042168
4,0.08,-0.042168


## Intégration de l'ENMO par epoch

Afin d'étudier l'activité physique sur différentes durées, les valeurs d'ENMO sont intégrées sur des epochs de 10, 30 et 60 secondes.

Une fonction `calculate_integrated_enmo` est créée afin de calculer l'ENMO intégré sur des epochs de différentes durées.

Elle prend en entrée :
- `df` : le DataFrame contenant les valeurs d'ENMO ;
- `epoch` : la durée de l'epoch en secondes (10, 30 ou 60 s) ;
- `fs` : la fréquence d'échantillonnage en Hz.

La fréquence d'échantillonnage et la durée de l'epoch permettent tout d'abord de déterminer le nombre d'échantillons contenus dans chaque intervalle (`fs × epoch`).

Les valeurs d'ENMO sont ensuite regroupées par epoch et additionnées. La somme obtenue est divisée par la fréquence d'échantillonnage afin de prendre en compte la durée de chaque échantillon et d'obtenir l'ENMO intégré en g·s.

Enfin, une division par 60 permet de convertir le résultat de g·s en g·min.

La fonction retourne une valeur d'ENMO intégré pour chaque epoch.

$$
\mathrm{ENMO}_{\mathrm{int}} =
\left(\sum_{i=1}^{N} \mathrm{ENMO}_i\right)
\times \frac{1}{f_s}
\times \frac{1}{60}
$$

In [ ]:
def calculate_integrated_enmo(df, epoch, fs):
    samples = int(fs * epoch)

    integrated_enmo = (
        df["enmo"]
        .groupby(df.index // samples)
        .sum()
        / fs
        / 60
    )

    return integrated_enmo

La fonction `calculate_integrated_enmo` est appliquée successivement à des epochs de 10, 30 et 60 secondes. Les résultats obtenus sont stockés dans trois variables distinctes afin de pouvoir les utiliser par la suite pour la visualisation.

Le nombre d'epochs obtenu pour chaque durée est ensuite affiché afin de vérifier le découpage des données. Il correspondra au nombre de barres présents sur le graphique représentant l'ENMO intégré (g.min) en fonction du temps (min).

In [11]:
integrated_enmo_10s = calculate_integrated_enmo(df, 10, fs)
integrated_enmo_30s = calculate_integrated_enmo(df, 30, fs)
integrated_enmo_60s = calculate_integrated_enmo(df, 60, fs)

print("Number of 10 s epochs:", len(integrated_enmo_10s))
print("Number of 30 s epochs:", len(integrated_enmo_30s))
print("Number of 60 s epochs:", len(integrated_enmo_60s))

Number of 10 s epochs: 42
Number of 30 s epochs: 14
Number of 60 s epochs: 7
